# 🏗️ ActiveMQ on Azure Container Apps with Azure Firewall - Deployment & Cost

## Architecture Overview

```mermaid
graph TB
    Internet["🌐 INTERNET"] --> FW

    subgraph VNet["VNet (10.0.0.0/16)"]
        subgraph FWSub["AzureFirewallSubnet (10.0.1.0/24)"]
            FW["🔥 Azure Firewall<br/>Public IP: SNAT<br/>Private IP: 10.0.1.4"]
        end

        subgraph ACASub["ACA Subnet (10.0.16.0/20)"]
            AMQ["📦 ActiveMQ<br/>:61616 (OpenWire)<br/>:8161 (Web Console)"]
        end
    end

    FW -->|"✅ Whitelisted IP<br/>DNAT → ActiveMQ"| AMQ
    FW -.->|"❌ Other IPs<br/>DROP"| Blocked["🚫 Dropped"]

    AMQ -->|"UDR: 0.0.0.0/0 → FW<br/>(All egress forced through FW)"| FW

    style Internet fill:#263238,color:#fff,stroke:#263238
    style VNet fill:#e3f2fd,stroke:#1565c0,stroke-width:2px
    style FWSub fill:#fff3e0,stroke:#ff9800,stroke-width:2px
    style ACASub fill:#e8f5e9,stroke:#4caf50,stroke-width:2px
    style FW fill:#ff9800,color:#fff,stroke:#e65100,stroke-width:2px
    style AMQ fill:#4caf50,color:#fff,stroke:#2e7d32,stroke-width:2px
    style Blocked fill:#ef5350,color:#fff,stroke:#c62828
```

## How Azure Firewall Egress IP (SNAT) Works

| Scenario | Egress IP Behavior |
|----------|-------------------|
| 1 Public IP | All outbound traffic always uses that single IP |
| Multiple Public IPs | Azure Firewall can SNAT using **any** of them (round-robin) |

**Key Points:**
- Azure Firewall performs **Source NAT (SNAT)** on all outbound internet traffic
- The Firewall's Public IP becomes the **egress identity** of your workload
- External services see traffic coming from the Firewall's Public IP, NOT the container's private IP
- This gives you a **stable, predictable egress IP** that partners/customers can whitelist

## 💰 Cost Estimation (Before Deployment)

Let's calculate the estimated monthly cost for this architecture.

In [ ]:
# =============================================================================
# Azure Cost Estimator for ActiveMQ + Firewall + Container Apps
# =============================================================================
# Prices based on East US 2 region (USD, pay-as-you-go)
# Source: https://azure.microsoft.com/en-us/pricing/
# =============================================================================

import json
from dataclasses import dataclass, field
from typing import List

@dataclass
class CostItem:
    service: str
    component: str
    unit: str
    quantity: float
    unit_price: float
    notes: str = ""
    
    @property
    def monthly_cost(self) -> float:
        return self.quantity * self.unit_price

# =============================================================================
# Define all cost components
# =============================================================================
costs: List[CostItem] = [
    # Azure Firewall Basic (Dev/Demo - much cheaper than Standard)
    CostItem(
        service="Azure Firewall",
        component="Deployment (Basic SKU)",
        unit="hours/month",
        quantity=730,        # 24x7
        unit_price=0.395,    # $0.395/hour (vs $1.25 for Standard)
        notes="Basic SKU for dev/demo. 250 Mbps. Upgrade to Standard for production."
    ),
    CostItem(
        service="Azure Firewall",
        component="Data Processing",
        unit="GB/month",
        quantity=100,        # Estimated 100 GB/month
        unit_price=0.016,    # $0.016/GB
        notes="Varies with traffic volume"
    ),
    
    # Public IP
    CostItem(
        service="Public IP",
        component="Standard Static IP (Firewall)",
        unit="hours/month",
        quantity=730,
        unit_price=0.005,    # ~$3.65/month
        notes="Static IP for Firewall egress/SNAT"
    ),
    CostItem(
        service="Public IP",
        component="Standard Static IP (Mgmt)",
        unit="hours/month",
        quantity=730,
        unit_price=0.005,    # ~$3.65/month
        notes="Management IP required for Basic SKU"
    ),
    
    # Container Apps
    CostItem(
        service="Container Apps",
        component="vCPU (1 core, always-on)",
        unit="vCPU-hours/month",
        quantity=730,        # 1 vCPU x 730 hours
        unit_price=0.000024, # Consumption plan per vCPU-second = $0.000024
        notes="1 vCPU for ActiveMQ container (consumption pricing per second)"
    ),
    CostItem(
        service="Container Apps",
        component="Memory (2 GiB, always-on)",
        unit="GiB-hours/month",
        quantity=1460,       # 2 GiB x 730 hours
        unit_price=0.000003, # Per GiB-second = $0.000003
        notes="2 GiB RAM for ActiveMQ"
    ),
    
    # Log Analytics
    CostItem(
        service="Log Analytics",
        component="Data Ingestion",
        unit="GB/month",
        quantity=5,          # ~5 GB of logs
        unit_price=2.76,     # $2.76/GB
        notes="First 5 GB/month free per subscription. Estimate for demo."
    ),
    
    # VNet
    CostItem(
        service="Virtual Network",
        component="VNet + Subnets",
        unit="flat",
        quantity=1,
        unit_price=0.00,
        notes="VNet itself is free. Peering/VPN Gateway would add cost."
    ),
]

# =============================================================================
# Print Cost Summary
# =============================================================================
print("=" * 90)
print("  AZURE COST ESTIMATION - ActiveMQ + Azure Firewall + Container Apps")
print("  Region: East US 2 | Pricing: Pay-As-You-Go | Date: 2026-03")
print("=" * 90)
print()

total = 0.0
current_service = ""

for item in costs:
    if item.service != current_service:
        current_service = item.service
        print(f"\n📦 {current_service}")
        print(f"  {'Component':<35} {'Qty':>10} {'Unit Price':>12} {'Monthly':>12}")
        print(f"  {'-'*35} {'-'*10} {'-'*12} {'-'*12}")
    
    monthly = item.monthly_cost
    total += monthly
    print(f"  {item.component:<35} {item.quantity:>10.1f} ${item.unit_price:>10.4f} ${monthly:>10.2f}")
    if item.notes:
        print(f"    ℹ️  {item.notes}")

print(f"\n{'=' * 90}")
print(f"  💰 ESTIMATED TOTAL MONTHLY COST:  ${total:>10.2f} /month")
print(f"  💰 ESTIMATED ANNUAL COST:         ${total * 12:>10.2f} /year")
print(f"{'=' * 90}")

print(f"\n⚠️  IMPORTANT NOTES:")
print(f"  • Azure Firewall Basic SKU (~$288/mo) - ideal for dev/demo")
print(f"  • Basic SKU: 250 Mbps throughput, DNAT, Network Rules, Threat Intel")
print(f"  • For production, upgrade to Standard (~$912/mo) for TLS inspection, IDPS")
print(f"  • Container Apps consumption pricing is very cost-effective")
print(f"  • Actual costs may vary based on traffic volume and scaling")
print(f"  • Free tier: Log Analytics first 5 GB/month, VNet is free")

## 🔧 Pre-Deployment Setup

Configure your deployment parameters below.

In [ ]:
# =============================================================================
# Deployment Configuration
# =============================================================================
import subprocess
import json
import os

# Configuration - MODIFY THESE VALUES
SUBSCRIPTION_ID = "<your-subscription-id>"     # az account show --query id -o tsv
RESOURCE_GROUP  = "rg-activemq-firewall-mvp"
LOCATION        = "eastus2"
BASE_NAME       = "activemq-mvp"

# Whitelisted IPs - ONLY these can reach ActiveMQ through the firewall
# Replace with your actual client IPs
WHITELISTED_IPS = [
    "203.0.113.10",      # Example: Customer Office IP
    "198.51.100.0/24",   # Example: Partner Network CIDR
]

# Get current public IP for auto-whitelisting
def get_my_public_ip():
    """Get the current machine's public IP for whitelisting."""
    try:
        import urllib.request
        ip = urllib.request.urlopen('https://api.ipify.org').read().decode('utf8')
        return ip
    except Exception:
        return None

my_ip = get_my_public_ip()
if my_ip:
    print(f"✅ Your current public IP: {my_ip}")
    if my_ip not in WHITELISTED_IPS:
        WHITELISTED_IPS.append(my_ip)
        print(f"   Added to whitelist automatically")
else:
    print("⚠️  Could not detect public IP. Add your IP to WHITELISTED_IPS manually.")

print(f"\n📋 Deployment Configuration:")
print(f"   Resource Group:  {RESOURCE_GROUP}")
print(f"   Location:        {LOCATION}")
print(f"   Base Name:       {BASE_NAME}")
print(f"   Whitelisted IPs: {WHITELISTED_IPS}")

## 🔐 Azure Login & Subscription Setup

In [ ]:
# =============================================================================
# Step 1: Verify Azure CLI login and set subscription
# =============================================================================

def run_az(cmd: str, check=True) -> dict:
    """Run an Azure CLI command and return parsed JSON output."""
    result = subprocess.run(
        f"az {cmd}",
        shell=True, capture_output=True, text=True
    )
    if check and result.returncode != 0:
        print(f"❌ Error: {result.stderr}")
        raise RuntimeError(result.stderr)
    try:
        return json.loads(result.stdout) if result.stdout.strip() else {}
    except json.JSONDecodeError:
        return {"raw": result.stdout}

# Check login
account = run_az("account show -o json")
print(f"✅ Logged in as: {account.get('user', {}).get('name', 'N/A')}")
print(f"   Subscription: {account.get('name', 'N/A')} ({account.get('id', 'N/A')})")

if SUBSCRIPTION_ID != "<your-subscription-id>":
    run_az(f"account set --subscription {SUBSCRIPTION_ID}")
    print(f"   ✅ Switched to subscription: {SUBSCRIPTION_ID}")
else:
    SUBSCRIPTION_ID = account.get('id')
    print(f"   Using current subscription: {SUBSCRIPTION_ID}")

## 🚀 Deploy Infrastructure

In [ ]:
# =============================================================================
# Step 2: Create Resource Group
# =============================================================================

result = run_az(f"group create --name {RESOURCE_GROUP} --location {LOCATION} -o json")
print(f"✅ Resource Group: {result.get('name', RESOURCE_GROUP)} ({result.get('properties', {}).get('provisioningState', 'OK')})")

In [ ]:
# =============================================================================
# Step 3: Deploy Bicep Template
# =============================================================================
# This deploys: VNet, Azure Firewall, Container Apps Environment, ActiveMQ
# Expected deployment time: ~15-20 minutes (Firewall takes ~10 min)
# =============================================================================

import time

# Build parameters JSON
params = {
    "location": {"value": LOCATION},
    "baseName": {"value": BASE_NAME},
    "whitelistedIPs": {"value": WHITELISTED_IPS},
}
params_json = json.dumps(params)

print("🚀 Starting deployment (this will take ~15-20 minutes)...")
print(f"   Firewall alone takes ~10 minutes to provision.\n")

start = time.time()

# Path to Bicep file - adjust if needed
bicep_path = os.path.join(os.path.dirname(os.getcwd()), "infra", "main.bicep")
if not os.path.exists(bicep_path):
    bicep_path = "infra/main.bicep"  # fallback relative path

deploy_cmd = (
    f"deployment group create "
    f"--resource-group {RESOURCE_GROUP} "
    f"--template-file {bicep_path} "
    f"--parameters '{params_json}' "
    f"--name activemq-deploy-{int(time.time())} "
    f"-o json"
)

result = run_az(deploy_cmd)
elapsed = time.time() - start

state = result.get('properties', {}).get('provisioningState', 'Unknown')
outputs = result.get('properties', {}).get('outputs', {})

print(f"\n{'='*60}")
print(f"  Deployment Status: {state}")
print(f"  Duration: {elapsed/60:.1f} minutes")
print(f"{'='*60}")

if outputs:
    FIREWALL_PUBLIC_IP = outputs.get('firewallPublicIp', {}).get('value', 'N/A')
    FIREWALL_PRIVATE_IP = outputs.get('firewallPrivateIp', {}).get('value', 'N/A')
    ACTIVEMQ_FQDN = outputs.get('activeMqFqdn', {}).get('value', 'N/A')
    
    print(f"\n📋 Deployment Outputs:")
    print(f"   Firewall Public IP (Egress/SNAT):  {FIREWALL_PUBLIC_IP}")
    print(f"   Firewall Private IP:               {FIREWALL_PRIVATE_IP}")
    print(f"   ActiveMQ Internal FQDN:            {ACTIVEMQ_FQDN}")
    print(f"\n   🔑 External clients connect to: {FIREWALL_PUBLIC_IP}:61616")
    print(f"   🌐 Web Console (if DNAT'd):       http://{FIREWALL_PUBLIC_IP}:8161")

## ✅ Verify Deployment

In [ ]:
# =============================================================================
# Step 4: Verify all resources are deployed correctly
# =============================================================================

resources = run_az(f"resource list --resource-group {RESOURCE_GROUP} -o json")

print(f"\n📦 Resources in {RESOURCE_GROUP}:")
print(f"{'Name':<45} {'Type':<45} {'Status'}")
print(f"{'-'*45} {'-'*45} {'-'*15}")

for r in resources:
    name = r.get('name', 'N/A')
    rtype = r.get('type', 'N/A')
    state = r.get('provisioningState', 'N/A')
    icon = '✅' if state == 'Succeeded' else '⏳'
    print(f"{icon} {name:<43} {rtype:<45} {state}")

# Verify Firewall rules
print(f"\n🔥 Firewall Rules:")
fw_policy = run_az(
    f"network firewall policy rule-collection-group list "
    f"--policy-name {BASE_NAME}-fw-policy "
    f"--resource-group {RESOURCE_GROUP} -o json",
    check=False
)
if isinstance(fw_policy, list):
    for group in fw_policy:
        for collection in group.get('ruleCollections', []):
            print(f"   📋 {collection.get('name', 'N/A')} (priority: {collection.get('priority', 'N/A')})")
            for rule in collection.get('rules', []):
                print(f"      → {rule.get('name', 'N/A')}: {rule.get('description', '')}")

## 🧹 Cleanup (When Done)

⚠️ **Only run this when you want to delete ALL resources to stop costs.**

In [ ]:
# =============================================================================
# CLEANUP - Delete everything to stop costs
# =============================================================================
# ⚠️  UNCOMMENT AND RUN ONLY WHEN YOU WANT TO DELETE EVERYTHING
# =============================================================================

# CONFIRM_DELETE = False  # Change to True to enable deletion

# if CONFIRM_DELETE:
#     print(f"🗑️  Deleting resource group: {RESOURCE_GROUP}...")
#     run_az(f"group delete --name {RESOURCE_GROUP} --yes --no-wait")
#     print(f"✅ Deletion initiated (takes ~5-10 minutes in background)")
# else:
#     print("⚠️  Set CONFIRM_DELETE = True to delete resources")